# Week 5 — Spark Fundamentals: Data Cleaning, Transformation & Aggregation


## Q1. Limitations of MapReduce vs Advantages of Spark

**Key limitations of traditional MapReduce:**
1. **Disk I/O overhead** — every Map and Reduce stage reads from and writes back to disk (HDFS), which is slow for multi-step/iterative jobs.
2. **No in-memory computation** — intermediate results cannot be cached in memory between stages.
3. **High latency for iterative algorithms** — machine learning / graph algorithms that need many passes over the same data become very expensive.
4. **Rigid two-stage programming model** — only Map and Reduce phases; complex pipelines need many chained MapReduce jobs.
5. **Verbose, low-level API** — harder to express joins, filters, and aggregations compared to Spark's DataFrame/SQL API.
6. **No interactive/streaming support** — MapReduce is batch-oriented only.

**Why Spark is preferred:**
- **In-memory processing**: keeps data in RAM (as RDDs/DataFrames) across operations, avoiding repeated disk reads/writes.
- **DAG execution engine**: builds an optimized execution plan (Catalyst optimizer) instead of forcing rigid Map/Reduce stages.
- **Unified API**: batch, streaming, SQL, ML, and graph processing in one engine.
- **Speed**: typically 10–100x faster than MapReduce for iterative workloads because of in-memory caching.

## Q2. In-Memory Computing and Iterative ML Algorithms

Disk-based systems like classic MapReduce write intermediate output to HDFS after every stage and reload it for the next stage. For an iterative algorithm (e.g., gradient descent, k-means) that touches the same dataset dozens or hundreds of times, this means dozens of disk read/write round-trips.

Spark instead lets you **cache/persist** a DataFrame or RDD in memory (`.cache()` / `.persist()`) once, then reuse it across iterations directly from RAM. Combined with **lazy evaluation** and the **DAG scheduler**, Spark can also pipeline multiple transformations together and only materialize results when an action is triggered — reducing unnecessary disk traffic even further. This is why Spark MLlib's iterative algorithms (logistic regression, k-means, ALS) run dramatically faster than disk-based MapReduce equivalents.

## Setup: Spark Session and Sample Dataset

Before answering the coding questions, we create a `SparkSession` and a sample **sales** dataset that intentionally contains duplicates, nulls, and empty strings — so every cleaning operation below has something real to act on.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType
)

spark = (
    SparkSession.builder
    .appName("Week5-Spark")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")  # small for local demo
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 4.0.2


In [2]:
# Sample raw data: includes duplicate rows, nulls, and empty strings on purpose
data = [
    (1, "u101", "2024-01-05", "West",  "Electronics", 250.0, 28, "Premium", "a@mail.com", "alice",  "NYC",        "2024-01-05 10:00:00"),
    (2, "u102", "2024-01-06", "East",  "Grocery",     45.5,  35, "Basic",   "b@mail.com", "bob",    "Boston",     "2024-01-06 11:15:00"),
    (3, "u103", "2024-01-07", "West",  "Electronics", None,  22, "Premium", "c@mail.com", "carol",  "LA",         "2024-01-07 09:30:00"),
    (4, "u104", "2024-01-08", "North", "Apparel",     120.0, 41, "Basic",   None,         "dave",   "Chicago",    "2024-01-08 14:45:00"),
    (5, "u105", "2024-01-09", "West",  "Grocery",     60.0,  19, "Premium", "e@mail.com", "",       "NYC",        "2024-01-09 08:20:00"),
    (6, "u101", "2024-01-05", "West",  "Electronics", 250.0, 28, "Premium", "a@mail.com", "alice",  "NYC",        "2024-01-05 10:00:00"),  # duplicate of row 1
    (7, "u106", "2024-01-10", "South", "Apparel",     None,  None, "Basic", "g@mail.com", "grace",  "Miami",      "2024-01-10 16:00:00"),
    (8, "u107", "2024-01-11", "West",  "Electronics", 300.0, 30, "Premium", "h@mail.com", "heidi",  "NYC",        "2024-01-11 13:10:00"),
    (9, "u108", "2024-01-12", "East",  "Grocery",     55.0,  65, "Basic",   "i@mail.com", "ivan",   "Boston",     "2024-01-12 17:30:00"),
    (10,"u109", "2024-01-13", "West",  "Apparel",     90.0,  24, "Premium", None,         "jack",   "NYC",        "2024-01-13 12:00:00"),
]

columns = ["order_id", "user_id", "transaction_date", "region", "product_category",
           "sale_amount", "age", "subscription", "email", "username", "city", "raw_timestamp"]

df_sales = spark.createDataFrame(data, columns)
print(f"Row count: {df_sales.count()}")
df_sales.show(truncate=False)

Row count: 10
+--------+-------+----------------+------+----------------+-----------+----+------------+----------+--------+-------+-------------------+
|order_id|user_id|transaction_date|region|product_category|sale_amount|age |subscription|email     |username|city   |raw_timestamp      |
+--------+-------+----------------+------+----------------+-----------+----+------------+----------+--------+-------+-------------------+
|1       |u101   |2024-01-05      |West  |Electronics     |250.0      |28  |Premium     |a@mail.com|alice   |NYC    |2024-01-05 10:00:00|
|2       |u102   |2024-01-06      |East  |Grocery         |45.5       |35  |Basic       |b@mail.com|bob     |Boston |2024-01-06 11:15:00|
|3       |u103   |2024-01-07      |West  |Electronics     |NULL       |22  |Premium     |c@mail.com|carol   |LA     |2024-01-07 09:30:00|
|4       |u104   |2024-01-08      |North |Apparel         |120.0      |41  |Basic       |NULL      |dave    |Chicago|2024-01-08 14:45:00|
|5       |u105   |20

## Q3. Remove Duplicate Rows Based on (user_id, transaction_date)

`dropDuplicates()` returns a **new** DataFrame (immutability — see Q7) with duplicate rows removed, keeping the first occurrence for each combination of the given subset of columns.

In [3]:
df_no_dupes = df_sales.dropDuplicates(["user_id", "transaction_date"])

print(f"Before: {df_sales.count()} rows -> After: {df_no_dupes.count()} rows")
df_no_dupes.orderBy("order_id").show(truncate=False)

Before: 10 rows -> After: 9 rows
+--------+-------+----------------+------+----------------+-----------+----+------------+----------+--------+-------+-------------------+
|order_id|user_id|transaction_date|region|product_category|sale_amount|age |subscription|email     |username|city   |raw_timestamp      |
+--------+-------+----------------+------+----------------+-----------+----+------------+----------+--------+-------+-------------------+
|1       |u101   |2024-01-05      |West  |Electronics     |250.0      |28  |Premium     |a@mail.com|alice   |NYC    |2024-01-05 10:00:00|
|2       |u102   |2024-01-06      |East  |Grocery         |45.5       |35  |Basic       |b@mail.com|bob     |Boston |2024-01-06 11:15:00|
|3       |u103   |2024-01-07      |West  |Electronics     |NULL       |22  |Premium     |c@mail.com|carol   |LA     |2024-01-07 09:30:00|
|4       |u104   |2024-01-08      |North |Apparel         |120.0      |41  |Basic       |NULL      |dave    |Chicago|2024-01-08 14:45:00|
|

**Result/Insight:** Row 6 (a duplicate of row 1 on `user_id='u101'`, `transaction_date='2024-01-05'`) was dropped, reducing 10 rows to 9. The original `df_sales` is untouched.

## Q4. Filter region = 'West' then groupBy product_category, avg(sale_amount)

In [4]:
west_avg = (
    df_sales
    .filter(F.col("region") == "West")
    .groupBy("product_category")
    .agg(F.avg("sale_amount").alias("avg_sale_amount"))
)
west_avg.show(truncate=False)

+----------------+-----------------+
|product_category|avg_sale_amount  |
+----------------+-----------------+
|Electronics     |266.6666666666667|
|Grocery         |60.0             |
|Apparel         |90.0             |
+----------------+-----------------+



**Insight:** Spark's `null` in `sale_amount` (order_id 3) is automatically excluded from `avg()` — aggregation functions ignore nulls by default, which is why handling nulls *before* aggregating (see Q9) still matters for correctness of counts and totals.

## Q5. .na.drop() vs .na.fill() + filling nulls in `status`/subscription column

- **`.na.drop()`** removes rows that contain null values (optionally restricted to specific columns via `subset=[...]`, and with `how="any"`/`"all"`). It **shrinks the row count**.
- **`.na.fill()`** replaces null values with a specified default, **keeping the row count the same** — it only changes column contents.

Below, since our sample data doesn't have a `status` column, we demonstrate the same pattern by filling nulls in the `subscription` column with `'Unknown'` (the technique is identical for any string column named `status`).

In [5]:
# .na.drop() example - drop rows with ANY null
dropped = df_sales.na.drop()
print(f".na.drop() -> {dropped.count()} rows remain (out of {df_sales.count()})")

# .na.fill() example - fill nulls in a 'status'-like column ('subscription' here) with 'Unknown'
filled = df_sales.na.fill({"subscription": "Unknown"})
filled.select("order_id", "subscription").show()

# Equivalent generic pattern for an actual 'status' column:
# df.na.fill({"status": "Unknown"})

.na.drop() -> 6 rows remain (out of 10)
+--------+------------+
|order_id|subscription|
+--------+------------+
|       1|     Premium|
|       2|       Basic|
|       3|     Premium|
|       4|       Basic|
|       5|     Premium|
|       6|     Premium|
|       7|       Basic|
|       8|     Premium|
|       9|       Basic|
|      10|     Premium|
+--------+------------+



## Q6. Count Records per City, Keep Only Cities with Count > 100

Our demo dataset is tiny, so we lower the threshold to `> 1` just to show the mechanism producing a non-empty result; the **same `groupBy().count().filter()` pattern is exactly what you'd use with `> 100`** on a full-size dataset.

In [6]:
# Production pattern (use on the real, large dataset):
city_counts_100 = (
    df_sales.groupBy("city")
    .count()
    .filter(F.col("count") > 100)
)
print("Cities with count > 100 (none expected in this small sample):")
city_counts_100.show()

# Demonstration on our small sample with a lower threshold so we can see output
city_counts_demo = (
    df_sales.groupBy("city")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc())
)
print("Demo with threshold > 1 (to show non-empty result on sample data):")
city_counts_demo.show()

Cities with count > 100 (none expected in this small sample):
+----+-----+
|city|count|
+----+-----+
+----+-----+

Demo with threshold > 1 (to show non-empty result on sample data):
+------+-----+
|  city|count|
+------+-----+
|   NYC|    5|
|Boston|    2|
+------+-----+



**Insight:** `groupBy("city").count()` triggers a **shuffle** (see Q11) because rows for the same city may live on different partitions and must be brought together before counting.

## Q7. Immutability and Data Cleaning (drop/rename columns)

Spark DataFrames are **immutable** — no transformation (`drop`, `withColumnRenamed`, `filter`, `withColumn`, etc.) modifies the DataFrame in place. Each transformation returns a **brand-new DataFrame** representing the result, built lazily as a new node in the execution DAG. The original DataFrame reference is never changed.

Practical effects on cleaning workflows:
1. You must **reassign** the result of a cleaning step to a variable (often the same name) to "keep" the change, e.g. `df = df.drop("col")`.
2. It's safe to build **multiple cleaning pipelines from the same source DataFrame** without worrying about one step corrupting another's input — each branch is independent.
3. Cleaning steps are typically **chained** (`df.dropDuplicates().na.fill(...).withColumnRenamed(...)`) since each call just produces the next intermediate, immutable DataFrame.
4. Because of **lazy evaluation**, none of these intermediate DataFrames are actually computed until an action (`.show()`, `.count()`, `.write()`, etc.) is called — Spark optimizes the whole chain together via Catalyst.

In [7]:
# Demonstrating immutability: dropping a column does not change df_sales
df_dropped_email = df_sales.drop("email")

print("Original df_sales still has 'email'? ", "email" in df_sales.columns)
print("New df_dropped_email has 'email'?    ", "email" in df_dropped_email.columns)

Original df_sales still has 'email'?  True
New df_dropped_email has 'email'?     False


## Q8. Filter age 18–30 (inclusive) AND subscription = 'Premium'

In [8]:
filtered_age_sub = df_sales.filter(
    (F.col("age").between(18, 30)) & (F.col("subscription") == "Premium")
)
filtered_age_sub.select("order_id", "user_id", "age", "subscription").show()

+--------+-------+---+------------+
|order_id|user_id|age|subscription|
+--------+-------+---+------------+
|       1|   u101| 28|     Premium|
|       3|   u103| 22|     Premium|
|       5|   u105| 19|     Premium|
|       6|   u101| 28|     Premium|
|       8|   u107| 30|     Premium|
|      10|   u109| 24|     Premium|
+--------+-------+---+------------+



## Q9. Why Handle Nulls Before Aggregating?

1. **Silent exclusion vs. silent inflation/deflation**: `sum()` and `avg()` ignore nulls by default. If you don't explicitly decide how to treat them (drop vs. fill with 0 vs. fill with the mean), you may be unknowingly excluding records from the average's denominator, which **skews the result** without raising any error.
2. **Sum vs. average behave differently**: a null treated as missing (ignored) gives a different average than a null treated as 0 (counted in the denominator). Deciding this *before* aggregating avoids silently mixing both interpretations across pipeline stages.
3. **Downstream joins/groupBy keys**: a null in a **grouping key** column creates its own "null" group, which can fragment results or hide rows that should belong to a real group.
4. **Reproducibility**: aggregating over inconsistent null-handling makes results non-deterministic if the cleaning step is later changed, since the meaning of "missing" is then ambiguous in already-aggregated numbers.
5. **Performance**: handling nulls early (one pass) is cheaper than repeatedly re-deriving cleaned aggregates after each ad-hoc null-handling decision downstream.

## Q10. Cast raw_timestamp to TimestampType and Rename to event_time

In [9]:
df_event_time = (
    df_sales
    .withColumn("raw_timestamp", F.col("raw_timestamp").cast(TimestampType()))
    .withColumnRenamed("raw_timestamp", "event_time")
)

df_event_time.select("order_id", "event_time").printSchema()
df_event_time.select("order_id", "event_time").show(5, truncate=False)

root
 |-- order_id: long (nullable = true)
 |-- event_time: timestamp (nullable = true)

+--------+-------------------+
|order_id|event_time         |
+--------+-------------------+
|1       |2024-01-05 10:00:00|
|2       |2024-01-06 11:15:00|
|3       |2024-01-07 09:30:00|
|4       |2024-01-08 14:45:00|
|5       |2024-01-09 08:20:00|
+--------+-------------------+
only showing top 5 rows


## Q11. The Shuffle Process During Grouping (Wide Transformation)

When you call `groupBy(...).agg(...)`, Spark must collect all rows that share the same key into the same partition before it can compute the aggregate for that key. Because the relevant rows are usually scattered **across many partitions** (and across executors/nodes in a cluster), Spark has to:

1. **Repartition data by key** — each row is hashed on the grouping key(s) and routed to a target partition.
2. **Write intermediate ("shuffle") files to disk** on each executor and **transfer them over the network** to the executors that own the target partitions.
3. **Read and merge** the shuffled data on the receiving side before applying the final aggregation function.

This data movement across partition/executor boundaries is called the **shuffle**, and it is expensive (disk I/O + network I/O + serialization).

**Why it's a wide transformation:** A *narrow* transformation (like `filter` or `select`) can be computed independently, partition by partition, with no data movement — each output partition depends on only one input partition. A *wide* transformation (like `groupBy`, `join`, `distinct`, `repartition`) requires data from **multiple input partitions** to produce a single output partition, which forces the shuffle. `groupBy` is therefore a textbook wide transformation.

## Q12. Remove Rows Where email IS NULL OR username = '' (empty string)

In [10]:
df_clean_contacts = df_sales.filter(
    ~(F.col("email").isNull() | (F.col("username") == ""))
)

print(f"Before: {df_sales.count()} rows -> After: {df_clean_contacts.count()} rows")
df_clean_contacts.select("order_id", "email", "username").show()

Before: 10 rows -> After: 7 rows
+--------+----------+--------+
|order_id|     email|username|
+--------+----------+--------+
|       1|a@mail.com|   alice|
|       2|b@mail.com|     bob|
|       3|c@mail.com|   carol|
|       6|a@mail.com|   alice|
|       7|g@mail.com|   grace|
|       8|h@mail.com|   heidi|
|       9|i@mail.com|    ivan|
+--------+----------+--------+



**Insight:** Rows with `order_id` 4 (null email), 5 (empty username), 7 (null email) and 10 (null email) are removed.

## Q13. .agg() with Multiple Statistics at Once (min, max, mean of price)

We use `sale_amount` as our "price" column.

In [11]:
price_stats = df_sales.agg(
    F.min("sale_amount").alias("min_price"),
    F.max("sale_amount").alias("max_price"),
    F.mean("sale_amount").alias("mean_price"),
)
price_stats.show()

# Same idea per group (e.g., per product_category)
price_stats_by_category = df_sales.groupBy("product_category").agg(
    F.min("sale_amount").alias("min_price"),
    F.max("sale_amount").alias("max_price"),
    F.mean("sale_amount").alias("mean_price"),
)
price_stats_by_category.show()

+---------+---------+----------+
|min_price|max_price|mean_price|
+---------+---------+----------+
|     45.5|    300.0|  146.3125|
+---------+---------+----------+

+----------------+---------+---------+-----------------+
|product_category|min_price|max_price|       mean_price|
+----------------+---------+---------+-----------------+
|     Electronics|    250.0|    300.0|266.6666666666667|
|         Grocery|     45.5|     60.0|             53.5|
|         Apparel|     90.0|    120.0|            105.0|
+----------------+---------+---------+-----------------+



## Q14. Risk of `inferSchema=true` with Messy/Inconsistent Date Formats

When `inferSchema=true`, Spark samples the data and guesses a column's type based on what it sees. Risks when dates are messy or inconsistent:

1. **Silent fallback to StringType**: if Spark can't confidently parse a consistent date pattern across the sampled rows, it may infer the whole column as a plain string instead of `DateType`/`TimestampType` — so downstream date functions (`datediff`, `year()`, comparisons) won't work as expected without an explicit cast later.
2. **Silent nulls / corrupted values**: rows whose date string doesn't match the format Spark *did* infer can be silently converted to `null` (no error raised), causing **quiet data loss** that's easy to miss.
3. **Mixed format ambiguity**: formats like `MM/DD/YYYY` vs `DD/MM/YYYY` can be misinterpreted (e.g., `03/04/2024` parsed as March 4 vs April 3) with no warning, producing **logically wrong dates without any visible error**.
4. **Inference is based on a sample**: if the sample used for inference happens not to include the messiest rows, the inferred schema may look fine in testing but break (or silently mis-parse) in production on the full dataset.
5. **Performance cost**: `inferSchema=true` requires an **extra full (or sampled) pass over the data** before the real read, which is wasted cost if the format turns out to be unreliable anyway.

**Best practice:** explicitly define the schema (`StructType`) and parse dates with `to_date`/`to_timestamp` using a known format string, rather than relying on inference for messy source data.

## Q15. Final Processing Pipeline

Pipeline: filter out duplicates → fill null prices with 0 → group by `store_id` to calculate total revenue.

Since our sample dataset doesn't have a `store_id` column, we reuse `city` as the grouping key (the pattern is identical — swap in `store_id` against a real dataset) and `sale_amount` as "price".

In [12]:
final_pipeline_result = (
    df_sales
    .dropDuplicates(["user_id", "transaction_date"])      # 1. filter out duplicates
    .na.fill({"sale_amount": 0})                           # 2. fill null prices with 0
    .groupBy("city")                                       # 3. group by store_id (here: city)
    .agg(F.sum("sale_amount").alias("total_revenue"))
    .orderBy(F.col("total_revenue").desc())
)

final_pipeline_result.show()

# Equivalent code against a real dataset with a store_id column:
# final_pipeline_result = (
#     df_sales
#     .dropDuplicates(["user_id", "transaction_date"])
#     .na.fill({"price": 0})
#     .groupBy("store_id")
#     .agg(F.sum("price").alias("total_revenue"))
# )

+-------+-------------+
|   city|total_revenue|
+-------+-------------+
|    NYC|        700.0|
|Chicago|        120.0|
| Boston|        100.5|
|     LA|          0.0|
|  Miami|          0.0|
+-------+-------------+



## Summary of Insights

- **In-memory computing** is the core reason Spark outperforms disk-based MapReduce, especially for iterative workloads.
- **Immutability** means every cleaning step (drop, rename, filter) produces a new DataFrame; chains of these are cheap to build because of lazy evaluation and only materialize on an action.
- **Null-handling** (`.na.drop()` vs `.na.fill()`) should be a deliberate decision made *before* aggregation, since aggregate functions silently ignore nulls and this can distort sums/averages if not accounted for.
- **groupBy / join / distinct** are **wide transformations** that trigger a **shuffle** — the most expensive operation in a Spark job — because matching keys must be physically co-located across partitions.
- **inferSchema=true** is convenient but risky on messy data; explicit schemas + explicit `to_date`/`cast` calls are safer for production pipelines.
- A clean **pipeline pattern** (dedupe → fill/handle nulls → group/aggregate) is reusable, composable, and lets Spark's Catalyst optimizer plan the whole chain as one optimized job.

In [13]:
spark.stop()